# Enrich Labels with old Labels

## Import Libs

In [ ]:
import os
import geopandas as gpd
import pandas as pd
from pathlib import Path

## Define Paths

In [ ]:
notebook_dir = Path.cwd()
base_path = notebook_dir.parent.parent / "Data" 
base_path_str = str(base_path)
print(f"Base path automatically set to: {base_path_str}")

In [ ]:
label_dir = os.path.join(base_path, "9_Training_Data")
old_labels_path = os.path.join(label_dir, "training_data_old.shp")
gdf_old = gpd.read_file(old_labels_path)
new_labels_path = os.path.join(label_dir, "automated_training_labels_corrected.shp")
gdf_new = gpd.read_file(new_labels_path)

## Safety Check

In [ ]:
# CRS Safety Check: Ensure both dataframes use the same coordinate system
if gdf_new.crs != gdf_old.crs:
    print(" -> CRS mismatch! Transforming old labels to match new data...")
    gdf_old = gdf_old.to_crs(gdf_new.crs)

## Class Information

In [ ]:
print("\n--- OLD DATASET CLASS DISTRIBUTION ---")
old_counts = gdf_old['Class'].value_counts(dropna=False)
for cls_name, count in old_counts.items():
    print(f"Class '{cls_name}': {count} polygons")
print("--------------------------------------\n")

print("\n--- NEW DATASET CLASS DISTRIBUTION ---")
new_counts = gdf_new['Final_Clas'].value_counts(dropna=False)
for cls_name, count in new_counts.items():
    print(f"Class '{cls_name}': {count} polygons")
print("--------------------------------------\n")

## Derive Labels from old Label File

In [ ]:
print("Performing spatial join ('within')...")
# 'within' ensures the new polygon is strictly inside the old polygon boundaries
# Select only 'geometry' and 'Class' from the old dataset to keep it clean
joined_gdf = gpd.sjoin(gdf_new, gdf_old[['geometry', 'Class']], how='left', predicate='within')

# Deduplication: If a new polygon falls exactly on the border of two old polygons,
# sjoin creates duplicates. Keep only the first match to maintain data integrity.
joined_gdf = joined_gdf[~joined_gdf.index.duplicated(keep='first')]

# Create a strict mask ensuring two conditions:
# 1. The new polygon is currently unclassified (-1)
# 2. The old dataset actually provided a valid, non-NaN manual label
mask_transfer = (joined_gdf['Final_Clas'] == -1) & (joined_gdf['Class'].notna())

# Count and log how many unclassified polygons will inherit an old label
labels_found = mask_transfer.sum()
print(f" -> Successfully inherited {labels_found} old labels for unclassified polygons!")

# Transfer the old manual 'Class' to 'Final_Clas' only where the mask applies
joined_gdf.loc[mask_transfer, 'Final_Clas'] = joined_gdf.loc[mask_transfer, 'Class']

# Clean up data types: Convert column to integer to remove floating points (e.g., 20.0 -> 20)
joined_gdf['Final_Clas'] = joined_gdf['Final_Clas'].astype(int)

In [ ]:
print("\n--- NEW DATASET CLASS DISTRIBUTION  ---")
new_counts = joined_gdf['Final_Clas'].value_counts(dropna=False)
for cls_name, count in new_counts.items():
    display_name = "Unclassified (NaN)" if pd.isna(cls_name) else cls_name
    print(f"Class '{display_name}': {count} polygons")
print("----------------------------------------------------\n")

## New Class Information

## Save

In [ ]:
# Drop the temporary columns generated by the spatial join
final_output = joined_gdf.drop(columns=['index_right', 'Class', 'tree_cov', 'vine_cov'])

output_path = os.path.join(label_dir, "automated_training_labels_merged.shp")
final_output.to_file(output_path)
print(f"✅ Done! Saved final shapefile as: {output_path}")